# 03 — Classical CNN+LSTM baseline

A deliberately strong classical baseline: BatchNorm → Conv1d → ReLU → MaxPool → Dropout → LSTM →
dense head, trained with class-weighted BCE and AdamW. The eight-vector is treated as a
pseudo-sequence of length eight; the order is the PCA component index, **not** packet timing.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def show_image(path):
    """Render a saved PNG inside the notebook without extra display dependencies."""
    img = PILImage.open(path)
    plt.figure(figsize=(min(12, img.width / 100), min(8, img.height / 100)))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


print("Project root:", ROOT)

## 1. Test metrics (validation-selected threshold)

In [ ]:
metrics = json.loads((RESULTS / "metrics" / "cnn_lstm.json").read_text(encoding="utf-8"))
keys = ["threshold", "accuracy", "precision", "recall", "f1", "roc_auc", "fpr", "fnr", "mcc",
        "balanced_accuracy", "tn", "fp", "fn", "tp"]
pd.Series({k: metrics[k] for k in keys}).to_frame("CNN+LSTM")

The threshold shown here (0.35) was selected on the validation split by F1 and then frozen for the
common test set. The test split was not used to tune it.

## 2. Training behaviour

In [ ]:
show_image(RESULTS / "figures" / "cnn_lstm_training_curves.png")

In [ ]:
meta = json.loads((ROOT / "models" / "cnn_lstm_meta.json").read_text(encoding="utf-8"))
info = meta["info"]
pd.Series({k: info[k] for k in ["n_parameters", "epochs_run", "best_epoch",
                                "training_time_s", "device", "input_representation"]})

## 3. Confusion matrix and operating curves

In [ ]:
show_image(RESULTS / "figures" / "cnn_lstm_confusion_matrix.png")
show_image(RESULTS / "figures" / "cnn_lstm_roc_curve.png")
show_image(RESULTS / "figures" / "cnn_lstm_pr_curve.png")

## 4. Per-sample predictions

In [ ]:
pred = pd.read_csv(RESULTS / "predictions" / "cnn_lstm_test_predictions.csv")
print(pred.shape)
print(pred["model"].unique(), "threshold =", pred["threshold"].unique())
pred.head()

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(pred["y_true"], pred["y_pred"])
pd.DataFrame(cm, index=["true normal", "true attack"], columns=["pred normal", "pred attack"])

## 5. Threshold sensitivity

In [ ]:
show_image(RESULTS / "figures" / "cnn_lstm_threshold_analysis.png")
show_image(RESULTS / "figures" / "cnn_lstm_threshold_analysis_test.png")

**Takeaways**

1. CNN+LSTM is the strongest model on accuracy, F1, MCC and false-positive rate.
2. The selected threshold favours recall modestly while keeping FPR at 3.7%.
3. The baseline is not a straw man: it has ~9.7k parameters, early stopping and a tuned threshold.
4. Its pseudo-temporal input is a representation choice, not real packet timing.